#### Static

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import dgl
import dgl.nn as dglnn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import random

# ---------------------------
# Config / globals
# ---------------------------
CSV_PATH = "../all_circuits_features.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------------------
# Load & preprocess data
# ---------------------------
df = pd.read_csv(CSV_PATH)
df["gate_type"] = df["gate_type"].astype(str).str.strip().str.lower()

label_encoder = LabelEncoder()
df["gate_label"] = label_encoder.fit_transform(df["gate_type"])
classes_found = list(label_encoder.classes_)
class_to_index = {c: i for i, c in enumerate(classes_found)}

feature_columns = [
    "fan_in","fan_out","dist_to_output","is_primary_input","is_primary_output",
    "is_internal","is_key_gate","degree_centrality","betweenness_centrality",
    "closeness_centrality","clustering_coefficient","avg_fan_in_neighbors","avg_fan_out_neighbors"
]
df = df.dropna(subset=feature_columns)
df[feature_columns] = df[feature_columns].astype(float)
scaler = StandardScaler()
df[feature_columns] = scaler.fit_transform(df[feature_columns])

nodes = df["node"].tolist()
node_to_id = {node: i for i, node in enumerate(nodes)}

# Build graph (same heuristic)
potential_sources = df[df["fan_out"] > 0]["node"].tolist()
edges = []
for _, row in df.iterrows():
    node_id = node_to_id[row["node"]]
    k = int(row["fan_in"]) if not np.isnan(row["fan_in"]) else 0
    sources = potential_sources[:max(k, 0)]
    for src in sources:
        if src in node_to_id:
            edges.append((node_to_id[src], node_id))

if len(edges) == 0:
    raise ValueError("No edges found from heuristic; check fan_in/fan_out.")

src_nodes, dst_nodes = zip(*edges)
graph = dgl.graph((torch.tensor(src_nodes, dtype=torch.long),
                   torch.tensor(dst_nodes, dtype=torch.long)), num_nodes=len(nodes))
graph = dgl.add_self_loop(graph)
graph.ndata['features'] = torch.tensor(df[feature_columns].values, dtype=torch.float32)
graph.ndata['labels'] = torch.tensor(df["gate_label"].values, dtype=torch.long)
graph = graph.to(device)
graph.ndata['features'] = graph.ndata['features'].to(device)
graph.ndata['labels'] = graph.ndata['labels'].to(device)
full_labels = graph.ndata['labels']

edge_set = set((int(u.item()), int(v.item())) for u, v in zip(*graph.edges()))
in_feats = len(feature_columns)

# ---------------------------
# Incremental batches
# ---------------------------
incremental_batches = [
    ['and', 'not'],
    ['nor'],
    ['nand'],
    ['input'],
    ['output', 'or'],
    ['xor']
]
batch_label_indices = [[class_to_index[c] for c in batch] for batch in incremental_batches]

def split_for_batch(label_ids, test_frac=0.2, seed=SEED):
    rng = np.random.RandomState(seed)
    all_nodes = [node_to_id[n] for n, lbl in zip(nodes, df["gate_label"].values) if int(lbl) in label_ids]
    rng.shuffle(all_nodes)
    test_count = max(1, int(len(all_nodes) * test_frac))
    test_nodes = all_nodes[:test_count]
    train_nodes = all_nodes[test_count:]
    if len(train_nodes) == 0 and len(test_nodes) > 0: train_nodes.append(test_nodes.pop())
    if len(test_nodes) == 0 and len(train_nodes) > 0: test_nodes.append(train_nodes.pop())
    return train_nodes, test_nodes, all_nodes

tasks_node_sets = []
for batch_ids in batch_label_indices:
    tr, te, all_subset = split_for_batch(batch_ids, test_frac=0.2, seed=SEED)
    tasks_node_sets.append({"labels": batch_ids, "train_nodes": tr, "test_nodes": te, "all_nodes": all_subset})

# ---------------------------
# Static GIN encoder
# ---------------------------
class StaticGIN(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.gin1 = dglnn.GINConv(
            nn.Sequential(
                nn.Linear(in_feats, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            aggregator_type='sum'
        )
        self.gin2 = dglnn.GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            aggregator_type='sum'
        )
        self.dropout = nn.Dropout(dropout)
    def forward(self, g, features):
        h = self.gin1(g, features)
        h = torch.relu(h)
        h = self.dropout(h)
        h = self.gin2(g, h)
        h = torch.relu(h)
        return h

bce_logits_loss = nn.BCEWithLogitsLoss()

def link_raw_scores(node_embeds, edge_index_tensor):
    if edge_index_tensor.shape[0] == 0:
        return torch.empty((0,), device=node_embeds.device)
    u_idx = edge_index_tensor[:, 0]
    v_idx = edge_index_tensor[:, 1]
    return torch.sum(node_embeds[u_idx] * node_embeds[v_idx], dim=1)

def build_class_based_pos_neg(task_idx, max_edges=2000, seed=SEED):
    ts = tasks_node_sets[task_idx]
    nodes_subset = list(set(ts["all_nodes"]))
    batch_label_set = set(ts["labels"])
    pos = []
    for (u, v) in edge_set:
        if (u in nodes_subset) and (v in nodes_subset):
            lu = int(full_labels[u].item()); lv = int(full_labels[v].item())
            if (lu in batch_label_set) and (lv in batch_label_set):
                pos.append((u, v))
    pos = list(set(pos))
    if len(pos) == 0: return None
    if len(pos) > max_edges:
        random.Random(seed).shuffle(pos)
        pos = pos[:max_edges]
    negatives = []
    rng = random.Random(seed)
    target_neg = len(pos)
    attempts = 0
    while len(negatives) < target_neg and attempts < max(100000, 10*target_neg):
        u = rng.choice(nodes_subset); v = rng.choice(nodes_subset)
        if u != v and (u, v) not in edge_set:
            negatives.append((u, v))
        attempts += 1
    if len(negatives) < target_neg:
        candidates = [(u, v) for u in nodes_subset for v in nodes_subset if u != v and (u, v) not in edge_set]
        random.Random(seed+1).shuffle(candidates)
        for p in candidates:
            if p not in negatives:
                negatives.append(p)
            if len(negatives) >= target_neg: break
    pos_arr = np.array(pos, dtype=np.int64)
    neg_arr = np.array(negatives[:target_neg], dtype=np.int64)
    n = len(pos_arr)
    idx = np.arange(n)
    tr_idx, te_idx = train_test_split(idx, test_size=0.2, random_state=SEED)
    pos_tr, pos_te = pos_arr[tr_idx], pos_arr[te_idx]
    neg_tr, neg_te = neg_arr[tr_idx], neg_arr[te_idx]
    train_edges = np.vstack([pos_tr, neg_tr])
    train_labels = np.hstack([np.ones(len(pos_tr)), np.zeros(len(neg_tr))])
    test_edges = np.vstack([pos_te, neg_te])
    test_labels = np.hstack([np.ones(len(pos_te)), np.zeros(len(neg_te))])
    train_edges = torch.tensor(train_edges, dtype=torch.long, device=device)
    train_labels = torch.tensor(train_labels, dtype=torch.float32, device=device)
    test_edges = torch.tensor(test_edges, dtype=torch.long, device=device)
    test_labels = torch.tensor(test_labels, dtype=torch.float32, device=device)
    return train_edges, train_labels, test_edges, test_labels

# ---------------------------
# Mechanisms (Rehearsal, EWC, SI)
# ---------------------------
class RehearsalBufferEdges:
    def __init__(self, size=4000):
        self.size = size
        self.edges = []
        self.labels = []
    def add(self, edges_tensor, labels_tensor):
        e = edges_tensor.detach().cpu().numpy().tolist()
        l = labels_tensor.detach().cpu().numpy().tolist()
        self.edges.extend(e); self.labels.extend(l)
        seen = set()
        dedup_e, dedup_l = [], []
        for (u,v), lab in zip(self.edges, self.labels):
            key = (u,v,int(lab))
            if key in seen: continue
            seen.add(key); dedup_e.append([u,v]); dedup_l.append(lab)
        self.edges, self.labels = dedup_e, dedup_l
        if len(self.edges) > self.size:
            idx = np.random.choice(len(self.edges), self.size, replace=False)
            self.edges = [self.edges[i] for i in idx]
            self.labels = [self.labels[i] for i in idx]
    def get(self):
        if len(self.edges) == 0: return None
        edges_t = torch.tensor(self.edges, dtype=torch.long, device=device)
        labels_t = torch.tensor(self.labels, dtype=torch.float32, device=device)
        return edges_t, labels_t

class EWC:
    def __init__(self, model, lam=1000.0):
        self.model = model
        self.lam = lam
        self.star_vars = None
        self.fisher = None
    def compute_fisher_link(self, g, features, train_edges, train_labels, sample_size=2048):
        n = train_edges.shape[0]
        if n == 0: return
        sel = np.random.choice(n, min(sample_size, n), replace=False)
        e_sel = train_edges[sel]; l_sel = train_labels[sel]
        self.model.train()
        emb = self.model(g, features)
        logits = link_raw_scores(emb, e_sel)
        loss = bce_logits_loss(logits, l_sel)
        self.model.zero_grad(); loss.backward()
        self.fisher = [p.grad.detach().cpu().numpy()**2 if p.grad is not None else np.zeros_like(p.detach().cpu().numpy())
                       for p in self.model.parameters()]
        self.star_vars = [p.detach().cpu().numpy().copy() for p in self.model.parameters()]
    def penalty(self, model):
        if self.fisher is None or self.star_vars is None:
            return torch.tensor(0.0, device=device)
        total = 0.0
        for p, p_star, f in zip(model.parameters(), self.star_vars, self.fisher):
            W = p.detach().cpu().numpy()
            shape = tuple(min(a, b) for a, b in zip(W.shape, f.shape))
            slices = tuple(slice(0, s) for s in shape)
            W_s = W[slices]; p_star_s = p_star[slices]; f_s = f[slices]
            total += np.sum(f_s * (W_s - p_star_s)**2)
        return torch.tensor((self.lam/2.0)*total, dtype=torch.float32, device=device)

class SITracker:
    def __init__(self, model, c=5.0, eps=1e-3):
        self.model = model
        self.c = c
        self.eps = eps
        self.prev_vars = [p.detach().cpu().numpy().copy() for p in model.parameters()]
        self.w_importance = [np.zeros_like(p.detach().cpu().numpy()) for p in model.parameters()]
        self.path_int = [np.zeros_like(p.detach().cpu().numpy()) for p in model.parameters()]
    def update(self, grads):
        for i,(g,p) in enumerate(zip(grads,self.model.parameters())):
            if g is not None:
                delta = p.detach().cpu().numpy() - self.prev_vars[i]
                self.path_int[i] += (-g.detach().cpu().numpy()) * delta
            self.prev_vars[i] = p.detach().cpu().numpy().copy()
    def consolidate(self):
        for i,p in enumerate(self.model.parameters()):
            delta = p.detach().cpu().numpy() - self.prev_vars[i]
            self.w_importance[i] += self.path_int[i]/(delta**2+self.eps)
            self.path_int[i] = np.zeros_like(p.detach().cpu().numpy())
    def penalty(self, model):
        total=0.0
        for i,p in enumerate(model.parameters()):
            W = p.detach().cpu().numpy()
            prev = self.prev_vars[i]
            imp = self.w_importance[i]
            shape = tuple(min(a, b) for a, b in zip(W.shape, prev.shape))
            slices = tuple(slice(0, s) for s in shape)
            W_s = W[slices]; prev_s = prev[slices]; imp_s = imp[slices]
            total += np.sum(imp_s*(W_s - prev_s)**2)
        return torch.tensor(self.c*total, dtype=torch.float32, device=device)

# ---------------------------
# Catastrophic forgetting metrics
# ---------------------------
def weight_distance_metric(params_before, params_after):
    return sum(torch.norm(p0 - p1, p=2).item() for p0, p1 in zip(params_before, params_after))

def regularization_metric(params_before, params_after, fisher_list):
    total = 0.0
    for p0, p1, F in zip(params_before, params_after, fisher_list):
        a = p0.detach().cpu()
        b = p1.detach().cpu()
        Fa = torch.tensor(F, dtype=torch.float32)
        shape = tuple(min(x, y) for x, y in zip(a.shape, b.shape))
        slices = tuple(slice(0, s) for s in shape)
        total += torch.sum(Fa[slices] * (b[slices] - a[slices])**2).item()
    return total

def replay_effectiveness(acc_last, acc_first):
    return acc_last - acc_first

# ---------------------------
# Training/evaluation runner with metrics
# ---------------------------
def run_static_linkpred_gin_best(mechanism, cfg):
    model = StaticGIN(in_feats, cfg["hidden_dim"], dropout=cfg["dropout"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=0.0)

    buffer = RehearsalBufferEdges(size=cfg.get("buffer_size", 0)) if mechanism in ["rehearsal","gen_replay"] else None
    ewc = EWC(model, lam=cfg.get("ewc_lam", None)) if mechanism=="ewc" else None
    si = SITracker(model, c=cfg.get("si_lambda", None)) if mechanism=="si" else None

    params_before = [p.detach().clone() for p in model.parameters()]
    fisher_snapshot = [np.zeros_like(p.detach().cpu().numpy()) for p in model.parameters()]

    per_task_acc = []
    per_task_details = []

    for t_idx in range(len(incremental_batches)):
        data = build_class_based_pos_neg(t_idx, max_edges=cfg.get("max_edges_per_task", 2000), seed=SEED)
        if data is None:
            per_task_acc.append(float('nan'))
            per_task_details.append({"task_index": t_idx, "train_edges": 0})
            continue
        train_edges, train_labels, test_edges, test_labels = data

        # Train for given epochs
        for epoch in range(1, cfg["epochs"] + 1):
            model.train()
            optimizer.zero_grad()
            node_emb = model(graph, graph.ndata['features'])
            logits = link_raw_scores(node_emb, train_edges)
            tloss = bce_logits_loss(logits, train_labels)

            reg = torch.tensor(0.0, device=device)
            if ewc: reg = reg + ewc.penalty(model)
            if si:
                grads_tmp = torch.autograd.grad(tloss + reg, model.parameters(), retain_graph=True, allow_unused=True)
                si.update(grads_tmp)

            loss = tloss + reg
            loss.backward()
            optimizer.step()

        if si: si.consolidate()
        if ewc:
            ewc.compute_fisher_link(graph, graph.ndata['features'], train_edges, train_labels)
            fisher_snapshot = ewc.fisher  # keep last Fisher as importance proxy

        if buffer is not None:
            buffer.add(train_edges, train_labels)

        # Evaluate (binary acc) on test edges of current task
        with torch.no_grad():
            node_emb_eval = model(graph, graph.ndata['features'])
            scores = link_raw_scores(node_emb_eval, test_edges)
            probs = torch.sigmoid(scores).detach().cpu().numpy()
            preds_bin = (probs > 0.5).astype(int)
            y_true = test_labels.detach().cpu().numpy().astype(int)
            acc = (preds_bin == y_true).mean()
        per_task_acc.append(acc)
        cm = confusion_matrix(y_true, preds_bin, labels=[0,1])
        cr = classification_report(y_true, preds_bin, labels=[0,1], target_names=["no-edge","edge"], zero_division=0)
        per_task_details.append({
            "task_index": t_idx,
            "task_classes": incremental_batches[t_idx],
            "train_edges": int(train_edges.shape[0]),
            "test_edges": int(test_edges.shape[0]),
            "acc": acc,
            "cm": cm,
            "cr": cr
        })

    params_after = [p.detach().clone() for p in model.parameters()]
    wd = weight_distance_metric(params_before, params_after)
    reg_metric = regularization_metric(params_before, params_after, fisher_snapshot)
    replay_eff = None
    if mechanism in ["rehearsal","gen_replay"] and len(per_task_acc) >= 2:
        replay_eff = replay_effectiveness(per_task_acc[-1], per_task_acc[0])

    return {
        "mechanism": mechanism,
        "config": cfg,
        "task_accs": per_task_acc,
        "details": per_task_details,
        "weight_distance": wd,
        "regularization_metric": reg_metric,
        "replay_effectiveness": replay_eff
    }

# ---------------------------
# Best hyperparameters from your table (GIN Static, link prediction)
# ---------------------------
BEST_CONFIGS = {
    "naive": {
        "hidden_dim": 128,
        "dropout": 0.10,
        "epochs": 80,
        "lr": 3.937e-04
    },
    "rehearsal": {
        "hidden_dim": 32,
        "dropout": 0.10,
        "epochs": 80,
        "lr": 1.000e-03,
        "buffer_size": 4000
    },
    "ewc": {
        "hidden_dim": 64,
        "dropout": 0.30,
        "epochs": 180,
        "lr": 8.139e-04,
        "ewc_lam": 1000.0
    },
    "si": {
        "hidden_dim": 32,
        "dropout": 0.20,
        "epochs": 50,
        "lr": 1.000e-03,
        "si_lambda": 5.0
    },
    "gen_replay": {
        "hidden_dim": 256,
        "dropout": 0.10,
        "epochs": 80,
        "lr": 6.018e-04,
        "buffer_size": 4000
    }
}

# ---------------------------
# Run all mechanisms and print
# ---------------------------
results = []
for mech, cfg in BEST_CONFIGS.items():
    print(f"\n=== Running GIN Static LinkPred: {mech} ===")
    res = run_static_linkpred_gin_best(mech, cfg)
    results.append(res)
    print(f"- Task Accuracies: {np.round(res['task_accs'], 3).tolist()}")
    print(f"- Weight Distance Metric: {res['weight_distance']:.4f}")
    print(f"- Regularization Metric: {res['regularization_metric']:.4f}")
    if res['replay_effectiveness'] is not None:
        print(f"- Replay Effectiveness Metric: {res['replay_effectiveness']:.4f}")

# ---------------------------
# Save CSV summary
# ---------------------------
rows = []
for r in results:
    row = {
        "mechanism": r["mechanism"],
        "hidden_dim": r["config"]["hidden_dim"],
        "dropout": r["config"]["dropout"],
        "epochs": r["config"]["epochs"],
        "lr": r["config"]["lr"],
        "buffer_size": r["config"].get("buffer_size", "N/A"),
        "ewc_lam": r["config"].get("ewc_lam", "N/A"),
        "si_lambda": r["config"].get("si_lambda", "N/A"),
        "avg_acc": float(np.nanmean(r["task_accs"])),
        "batch1_acc": r["task_accs"][0] if len(r["task_accs"])>0 else np.nan,
        "batch2_acc": r["task_accs"][1] if len(r["task_accs"])>1 else np.nan,
        "batch3_acc": r["task_accs"][2] if len(r["task_accs"])>2 else np.nan,
        "batch4_acc": r["task_accs"][3] if len(r["task_accs"])>3 else np.nan,
        "batch5_acc": r["task_accs"][4] if len(r["task_accs"])>4 else np.nan,
        "batch6_acc": r["task_accs"][5] if len(r["task_accs"])>5 else np.nan,
        "weight_distance": r["weight_distance"],
        "regularization_metric": r["regularization_metric"],
        "replay_effectiveness": r["replay_effectiveness"] if r["replay_effectiveness"] is not None else "N/A"
    }
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary.to_csv("gin_static_linkpred_summary_with_cf_metrics.csv", index=False)
print("\nSaved gin_static_linkpred_summary_with_cf_metrics.csv")



=== Running GIN Static LinkPred: naive ===
- Task Accuracies: [0.652, 0.636, 0.681, 0.61, 0.725, 0.5]
- Weight Distance Metric: 9.0578
- Regularization Metric: 0.0000

=== Running GIN Static LinkPred: rehearsal ===
- Task Accuracies: [0.615, 0.658, 0.724, 0.614, 0.793, 0.783]
- Weight Distance Metric: 9.0833
- Regularization Metric: 0.0000
- Replay Effectiveness Metric: 0.1676

=== Running GIN Static LinkPred: ewc ===
- Task Accuracies: [0.812, 0.819, 0.872, 0.803, 0.905, 0.5]
- Weight Distance Metric: 14.8510
- Regularization Metric: 0.0022

=== Running GIN Static LinkPred: si ===
- Task Accuracies: [0.528, 0.581, 0.585, 0.594, 0.703, 0.804]
- Weight Distance Metric: 6.6626
- Regularization Metric: 0.0000

=== Running GIN Static LinkPred: gen_replay ===
- Task Accuracies: [0.834, 0.714, 0.764, 0.555, 0.823, 0.5]
- Weight Distance Metric: 13.9594
- Regularization Metric: 0.0000
- Replay Effectiveness Metric: -0.3337

Saved gin_static_linkpred_summary_with_cf_metrics.csv


In [2]:
# Cell 2: Dynamic GIN link prediction
# Assumes graph, device, incremental_batches, build_class_based_pos_neg, in_feats are defined in Cell 1.

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import dgl.nn as dglnn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

bce_logits_loss = nn.BCEWithLogitsLoss()

# ---------------------------
# Dynamic GIN encoder + per-task heads
# ---------------------------
class DynamicGIN(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.gin1 = dglnn.GINConv(
            nn.Sequential(
                nn.Linear(in_feats, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            aggregator_type='sum'
        )
        self.gin2 = dglnn.GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            aggregator_type='sum'
        )
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleList()
        self.hidden_dim = hidden_dim

    def add_task_head(self):
        self.heads.append(nn.Linear(self.hidden_dim, 1))

    def forward_emb(self, g, features):
        h = self.gin1(g, features)
        h = torch.relu(h)
        h = self.dropout(h)
        h = self.gin2(g, h)
        h = torch.relu(h)
        return h

    def forward_head(self, node_emb, task_index):
        return self.heads[task_index](node_emb).squeeze(-1)

# ---------------------------
# Link scoring
# ---------------------------
def link_raw_scores(node_embeds, edge_index_tensor):
    if edge_index_tensor.shape[0] == 0:
        return torch.empty((0,), device=node_embeds.device)
    u_idx = edge_index_tensor[:, 0]
    v_idx = edge_index_tensor[:, 1]
    return torch.sum(node_embeds[u_idx] * node_embeds[v_idx], dim=1)

# ---------------------------
# Mechanisms: replay, EWC, SI
# ---------------------------
class RehearsalBufferEdges:
    def __init__(self, size=4000):
        self.size = size
        self.edges = []; self.labels = []
    def add(self, edges_tensor, labels_tensor):
        e = edges_tensor.detach().cpu().numpy().tolist()
        l = labels_tensor.detach().cpu().numpy().tolist()
        self.edges.extend(e); self.labels.extend(l)
        seen = set(); dedup_e, dedup_l = [], []
        for (u,v), lab in zip(self.edges, self.labels):
            key = (u,v,int(lab))
            if key in seen: continue
            seen.add(key); dedup_e.append([u,v]); dedup_l.append(lab)
        self.edges, self.labels = dedup_e, dedup_l
        if len(self.edges) > self.size:
            idx = np.random.choice(len(self.edges), self.size, replace=False)
            self.edges = [self.edges[i] for i in idx]
            self.labels = [self.labels[i] for i in idx]
    def get(self):
        if len(self.edges) == 0: return None
        edges_t = torch.tensor(self.edges, dtype=torch.long, device=device)
        labels_t = torch.tensor(self.labels, dtype=torch.float32, device=device)
        return edges_t, labels_t

class EWC:
    def __init__(self, model, lam=1000.0):
        self.model = model; self.lam = lam
        self.star_vars = None; self.fisher = None
    def reset(self, model):
        self.model = model; self.star_vars = None; self.fisher = None
    def compute_fisher_link(self, g, features, train_edges, train_labels, sample_size=2048):
        n = train_edges.shape[0]
        if n == 0: return
        sel = np.random.choice(n, min(sample_size, n), replace=False)
        e_sel = train_edges[sel]; l_sel = train_labels[sel]
        self.model.train()
        emb = self.model.forward_emb(g, features)
        logits = link_raw_scores(emb, e_sel)
        loss = bce_logits_loss(logits, l_sel)
        self.model.zero_grad(); loss.backward()
        self.fisher = [p.grad.detach().cpu().numpy()**2 if p.grad is not None else np.zeros_like(p.detach().cpu().numpy())
                       for p in self.model.parameters()]
        self.star_vars = [p.detach().cpu().numpy().copy() for p in self.model.parameters()]
    def penalty(self, model):
        if self.fisher is None or self.star_vars is None:
            return torch.tensor(0.0, device=device)
        total = 0.0
        for p, p_star, f in zip(model.parameters(), self.star_vars, self.fisher):
            W = p.detach().cpu().numpy()
            shape = tuple(min(a, b) for a, b in zip(W.shape, f.shape))
            slices = tuple(slice(0, s) for s in shape)
            W_s = W[slices]; p_star_s = p_star[slices]; f_s = f[slices]
            total += np.sum(f_s * (W_s - p_star_s)**2)
        return torch.tensor((self.lam/2.0)*total, dtype=torch.float32, device=device)

class SITracker:
    def __init__(self, model, c=5.0, eps=1e-3):
        self.model = model; self.c = c; self.eps = eps
        self.prev_vars = [p.detach().cpu().numpy().copy() for p in model.parameters()]
        self.w_importance = [np.zeros_like(p.detach().cpu().numpy()) for p in model.parameters()]
        self.path_int = [np.zeros_like(p.detach().cpu().numpy()) for p in model.parameters()]
    def _sync_params(self):
        params = list(self.model.parameters())
        if len(params) > len(self.prev_vars):
            for p in params[len(self.prev_vars):]:
                W = p.detach().cpu().numpy()
                self.prev_vars.append(W.copy())
                self.w_importance.append(np.zeros_like(W))
                self.path_int.append(np.zeros_like(W))
        elif len(params) < len(self.prev_vars):
            self.prev_vars = self.prev_vars[:len(params)]
            self.w_importance = self.w_importance[:len(params)]
            self.path_int = self.path_int[:len(params)]
    def update(self, grads):
        self._sync_params()
        for i,(g,p) in enumerate(zip(grads,self.model.parameters())):
            if g is not None:
                delta = p.detach().cpu().numpy() - self.prev_vars[i]
                self.path_int[i] += (-g.detach().cpu().numpy()) * delta
            self.prev_vars[i] = p.detach().cpu().numpy().copy()
    def consolidate(self):
        self._sync_params()
        for i,p in enumerate(self.model.parameters()):
            delta = p.detach().cpu().numpy() - self.prev_vars[i]
            self.w_importance[i] += self.path_int[i]/(delta**2+self.eps)
            self.path_int[i] = np.zeros_like(p.detach().cpu().numpy())
    def penalty(self, model):
        self._sync_params()
        total=0.0
        for i,p in enumerate(model.parameters()):
            W = p.detach().cpu().numpy()
            prev = self.prev_vars[i]
            imp = self.w_importance[i]
            shape = tuple(min(a, b) for a, b in zip(W.shape, prev.shape))
            slices = tuple(slice(0, s) for s in shape)
            W_s = W[slices]; prev_s = prev[slices]; imp_s = imp[slices]
            total += np.sum(imp_s*(W_s - prev_s)**2)
        return torch.tensor(self.c*total, dtype=torch.float32, device=device)

# ---------------------------
# Metric helpers
# ---------------------------
def binary_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return acc, pr, rc, f1

def evaluate_edges(model, edges, labels):
    model.eval()
    with torch.no_grad():
        node_emb = model.forward_emb(graph, graph.ndata['features'])
        logits = link_raw_scores(node_emb, edges)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs > 0.5).astype(int)
        y_true = labels.detach().cpu().numpy().astype(int)
    return binary_metrics(y_true, preds)

# ---------------------------
# Catastrophic forgetting metrics
# ---------------------------
def weight_distance_metric(params_before, params_after):
    return sum(torch.norm(p0 - p1, p=2).item() for p0, p1 in zip(params_before, params_after))

def regularization_metric(params_before, params_after, fisher_list):
    total = 0.0
    for p0, p1, F in zip(params_before, params_after, fisher_list):
        a = p0.detach().cpu()
        b = p1.detach().cpu()
        Fa = torch.tensor(F, dtype=torch.float32)
        shape = tuple(min(x, y) for x, y in zip(a.shape, b.shape))
        slices = tuple(slice(0, s) for s in shape)
        total += torch.sum(Fa[slices] * (b[slices] - a[slices])**2).item()
    return total

# ---------------------------
# Best hyperparameters per mechanism (from your table)
# ---------------------------
BEST_CONFIGS = {
    "naive":      {"hidden_dim": 32,  "dropout": 0.20, "epochs": 150, "lr": 2.135e-03, "weight_decay": 0.0},
    "rehearsal":  {"hidden_dim": 128, "dropout": 0.30, "epochs": 120, "lr": 1.000e-03, "weight_decay": 0.0, "buffer_size": 4000},
    "ewc":        {"hidden_dim": 128, "dropout": 0.00, "epochs": 50,  "lr": 1.000e-03, "weight_decay": 0.0, "ewc_lam": 1000.0},
    "si":         {"hidden_dim": 64,  "dropout": 0.10, "epochs": 50,  "lr": 1.000e-03, "weight_decay": 0.0, "si_c": 5.0},
    "gen_replay": {"hidden_dim": 64,  "dropout": 0.00, "epochs": 80,  "lr": 1.449e-03, "weight_decay": 0.0, "buffer_size": 4000},
}

# ---------------------------
# Runner: train sequentially and report per-batch metrics + CF metrics
# ---------------------------
def run_dynamic_linkpred_gin(mechanism, cfg):
    model = DynamicGIN(in_feats, cfg["hidden_dim"], dropout=cfg["dropout"]).to(device)
    buffer = RehearsalBufferEdges(size=cfg.get("buffer_size", 0)) if mechanism in ["rehearsal","gen_replay"] else None
    ewc = EWC(model, lam=cfg.get("ewc_lam", None)) if mechanism=="ewc" else None
    si = SITracker(model, c=cfg.get("si_c", None)) if mechanism=="si" else None

    per_batch = []
    first_batch_test_acc = None

    for t_idx in range(len(incremental_batches)):
        # Add a new task head
        model.add_task_head()
        if si: si._sync_params()
        if ewc: ewc.reset(model)

        data = build_class_based_pos_neg(t_idx, max_edges=2000, seed=42)
        if data is None:
            per_batch.append({
                "task_index": t_idx, "classes": incremental_batches[t_idx],
                "train_acc": np.nan, "train_precision": np.nan, "train_recall": np.nan, "train_f1": np.nan,
                "test_acc": np.nan,  "test_precision": np.nan,  "test_recall": np.nan,  "test_f1": np.nan,
                "weight_distance": np.nan, "regularization_metric": np.nan, "replay_effectiveness": np.nan,
                "train_edges": 0, "test_edges": 0
            })
            continue

        train_edges, train_labels, test_edges, test_labels = data
        optimizer = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

        # Snapshot before training this batch
        params_before = [p.detach().clone() for p in model.parameters()]

        # Train epochs
        for epoch in range(1, cfg["epochs"] + 1):
            model.train()
            optimizer.zero_grad()
            node_emb = model.forward_emb(graph, graph.ndata['features'])
            logits = link_raw_scores(node_emb, train_edges)
            tloss = bce_logits_loss(logits, train_labels)

            reg = torch.tensor(0.0, device=device)
            if ewc: reg = reg + ewc.penalty(model)
            if si:
                grads_tmp = torch.autograd.grad(tloss + reg, model.parameters(), retain_graph=True, allow_unused=True)
                si.update(grads_tmp)

            loss = tloss + reg
            loss.backward()
            optimizer.step()

        # Update regularizers and buffer
        if si: si.consolidate()
        fisher_snapshot = None
        if ewc:
            ewc.compute_fisher_link(graph, graph.ndata['features'], train_edges, train_labels)
            fisher_snapshot = ewc.fisher
        if buffer is not None:
            buffer.add(train_edges, train_labels)

        # Train/test metrics
        train_acc, train_pr, train_rc, train_f1 = evaluate_edges(model, train_edges, train_labels)
        test_acc,  test_pr,  test_rc,  test_f1  = evaluate_edges(model, test_edges,  test_labels)

        if first_batch_test_acc is None:
            first_batch_test_acc = test_acc

        # Snapshot after training this batch
        params_after = [p.detach().clone() for p in model.parameters()]
        wd = weight_distance_metric(params_before, params_after)
        fisher_list = fisher_snapshot if fisher_snapshot is not None else [np.zeros_like(p.detach().cpu().numpy()) for p in params_before]
        reg_metric = regularization_metric(params_before, params_after, fisher_list)

        replay_eff = None
        if mechanism in ["rehearsal","gen_replay"]:
            replay_eff = test_acc - first_batch_test_acc

        # Record and print
        rec = {
            "task_index": t_idx,
            "classes": incremental_batches[t_idx],
            "train_acc": train_acc, "train_precision": train_pr, "train_recall": train_rc, "train_f1": train_f1,
            "test_acc": test_acc,   "test_precision": test_pr,   "test_recall": test_rc,   "test_f1": test_f1,
            "weight_distance": wd, "regularization_metric": reg_metric,
            "replay_effectiveness": replay_eff,
            "train_edges": int(train_edges.shape[0]), "test_edges": int(test_edges.shape[0]),
        }
        per_batch.append(rec)

        print(f"Batch {t_idx} {rec['classes']} | "
              f"Train Acc={train_acc:.3f} Prec={train_pr:.3f} Rec={train_rc:.3f} F1={train_f1:.3f} | "
              f"Test Acc={test_acc:.3f} Prec={test_pr:.3f} Rec={test_rc:.3f} F1={test_f1:.3f} | "
              f"WD={wd:.4f} Reg={reg_metric:.4f} ReplayEff={replay_eff if replay_eff is not None else 'N/A'} | "
              f"TrainEdges={rec['train_edges']} TestEdges={rec['test_edges']}")

    return per_batch

# ---------------------------
# Run all mechanisms and save CSV
# ---------------------------
all_results = {}
for mech, cfg in BEST_CONFIGS.items():
    print(f"\n=== Running GIN Dynamic LinkPred: {mech} ===")
    all_results[mech] = run_dynamic_linkpred_gin(mech, cfg)

# Save summary CSV
import pandas as pd
rows = []
for mech, res in all_results.items():
    for r in res:
        rows.append({
            "mechanism": mech,
            "task_index": r["task_index"],
            "classes": ",".join(r["classes"]),
            "train_acc": r["train_acc"], "train_precision": r["train_precision"], "train_recall": r["train_recall"], "train_f1": r["train_f1"],
            "test_acc": r["test_acc"],   "test_precision": r["test_precision"],   "test_recall": r["test_recall"],   "test_f1": r["test_f1"],
            "weight_distance": r["weight_distance"],
            "regularization_metric": r["regularization_metric"],
            "replay_effectiveness": r["replay_effectiveness"] if r["replay_effectiveness"] is not None else "N/A",
            "train_edges": r["train_edges"], "test_edges": r["test_edges"]
        })
df_summary = pd.DataFrame(rows)
df_summary.to_csv("gin_dynamic_linkpred_per_batch_metrics_with_cf.csv", index=False)
print("\nSaved gin_dynamic_linkpred_per_batch_metrics_with_cf.csv")



=== Running GIN Dynamic LinkPred: naive ===
Batch 0 ['and', 'not'] | Train Acc=0.890 Prec=0.820 Rec=0.999 F1=0.901 | Test Acc=0.895 Prec=0.826 Rec=1.000 F1=0.905 | WD=11.3350 Reg=0.0000 ReplayEff=N/A | TrainEdges=3200 TestEdges=800
Batch 1 ['nor'] | Train Acc=0.867 Prec=0.789 Rec=1.000 F1=0.882 | Test Acc=0.861 Prec=0.783 Rec=1.000 F1=0.878 | WD=5.9287 Reg=0.0000 ReplayEff=N/A | TrainEdges=3200 TestEdges=800
Batch 2 ['nand'] | Train Acc=0.900 Prec=0.842 Rec=0.986 F1=0.908 | Test Acc=0.884 Prec=0.819 Rec=0.985 F1=0.894 | WD=6.5696 Reg=0.0000 ReplayEff=N/A | TrainEdges=3200 TestEdges=800
Batch 3 ['input'] | Train Acc=0.615 Prec=0.566 Rec=0.993 F1=0.721 | Test Acc=0.620 Prec=0.569 Rec=0.994 F1=0.723 | WD=5.1642 Reg=0.0000 ReplayEff=N/A | TrainEdges=2902 TestEdges=726
Batch 4 ['output', 'or'] | Train Acc=0.844 Prec=0.763 Rec=0.999 F1=0.865 | Test Acc=0.824 Prec=0.741 Rec=0.997 F1=0.850 | WD=4.6551 Reg=0.0000 ReplayEff=N/A | TrainEdges=2818 TestEdges=706
Batch 5 ['xor'] | Train Acc=0.822 P

In [3]:
# Cell 3: Print catastrophic forgetting metrics only

for mech, res in all_results.items():
    print(f"\n=== Catastrophic Forgetting Metrics: {mech} ===")
    for r in res:
        print(f"Batch {r['task_index']} {r['classes']} | "
              f"WeightDist={r['weight_distance']:.4f} | "
              f"RegMetric={r['regularization_metric']:.4f} | "
              f"ReplayEff={r['replay_effectiveness'] if r['replay_effectiveness'] is not None else 'N/A'}")



=== Catastrophic Forgetting Metrics: naive ===
Batch 0 ['and', 'not'] | WeightDist=11.3350 | RegMetric=0.0000 | ReplayEff=N/A
Batch 1 ['nor'] | WeightDist=5.9287 | RegMetric=0.0000 | ReplayEff=N/A
Batch 2 ['nand'] | WeightDist=6.5696 | RegMetric=0.0000 | ReplayEff=N/A
Batch 3 ['input'] | WeightDist=5.1642 | RegMetric=0.0000 | ReplayEff=N/A
Batch 4 ['output', 'or'] | WeightDist=4.6551 | RegMetric=0.0000 | ReplayEff=N/A
Batch 5 ['xor'] | WeightDist=2.3792 | RegMetric=0.0000 | ReplayEff=N/A

=== Catastrophic Forgetting Metrics: rehearsal ===
Batch 0 ['and', 'not'] | WeightDist=10.4909 | RegMetric=0.0000 | ReplayEff=0.0
Batch 1 ['nor'] | WeightDist=6.3772 | RegMetric=0.0000 | ReplayEff=-0.12124999999999997
Batch 2 ['nand'] | WeightDist=7.3445 | RegMetric=0.0000 | ReplayEff=-0.043749999999999956
Batch 3 ['input'] | WeightDist=3.8258 | RegMetric=0.0000 | ReplayEff=-0.3440599173553719
Batch 4 ['output', 'or'] | WeightDist=4.8672 | RegMetric=0.0000 | ReplayEff=-0.03139164305949005
Batch 5 ['x